# SPY daily context and five-minute decisions
Build chronological chart pairs for every observed regular-session bar from 2020 through 2025. The left panel contains 42 completed daily candles. The right panel ends at the five-minute decision close. Both show EMA8, EMA21 and volume. The source and outcome files are separate. All timestamps are New York time. The first run downloads only the required historical partitions. Run all cells to build the six yearly PNG ZIPs and the decision index. No GPU is required for data preparation.

In [ ]:
%pip -q install "exchange-calendars==4.13.3" huggingface_hub pyarrow pillow
from pathlib import Path
import sys, requests, pandas as pd, zipfile, hashlib
root = Path('/content/spy_pairs')
root.mkdir(exist_ok=True, parents=True)
rev = '9969a58721cb49bb79e0826104759b4374a9e9e6'
base = f'https://raw.githubusercontent.com/jamelof23/SPY-EMA-V2/{rev}'
for path in ['src/collect_spy_minutes.py', 'src/paired_decision_charts.py', 'src/repair_paired_gaps.py', 'src/paired_setup_outcomes.py', 'data/source/SPY_daily_2019_2025.csv']:
    response = requests.get(f'{base}/{path}', timeout=60)
    response.raise_for_status()
    (root / Path(path).name).write_bytes(response.content)
sys.path.insert(0,str(root))
import collect_spy_minutes as source, paired_decision_charts as charts, repair_paired_gaps as gaps
from paired_setup_outcomes import call_candidates

In [ ]:
# Fetch monthly partitions and verify the primary feed against the exchange calendar.
minutes = root/'primary_minutes'
source.collect(minutes)
daily_file = root/'SPY_daily_2019_2025.csv'
_, _, _, _, before = charts.build(minutes, daily_file, root/'primary_audit')
days = before.index[before.expected != before.observed]
recovered, comparisons = gaps.repair(minutes, root/'reconciled_minutes', before, gaps.load_secondary(days))

In [ ]:
# Create the final dataset before rendering. Every outcome is keyed to its decision timestamp.
out = root/'dataset'
daily, bars, pairs, outcomes, audit = charts.build(root/'reconciled_minutes', daily_file, out)
recovered.to_parquet(out/'recovered_minutes.parquet',index=False)
comparisons.to_csv(out/'source_overlap_audit.csv',index=False)
before.to_parquet(out/'primary_session_audit.parquet')
restored = pd.to_datetime(recovered.timestamp,utc=True).dt.tz_convert(charts.ZONE).dt.tz_localize(None).dt.floor('5min')
bars['secondary_source_bar'] = bars.index.isin(restored)
pairs['secondary_in_42'] = bars.secondary_source_bar.rolling(42,min_periods=1).max().iloc[pairs.intraday_end].to_numpy().astype(bool)
pairs.to_parquet(out/'pairs.parquet',compression='zstd')
bars.to_parquet(out/'five_minute.parquet',compression='zstd')
candidates = call_candidates(daily,bars,pairs)
candidates.to_parquet(out/'call_setup_outcomes.parquet',compression='zstd')
print('Decision pairs by split:',pairs.groupby('split').size().to_dict())
print('First eligible CALL in each session:')
print(candidates.loc[candidates.first_eligible_signal_of_session].groupby(['split','outcome']).size())

### Review a decision
The example CALL uses the previous daily EMA8 as a fixed upside target and the previous close as its stop. It requires a green previous candle and a completed five-minute EMA8 cross above EMA21. Both levels must bracket the signal close and the next open. Other discretionary setups remain unlabeled. Forward outcomes are SPY price observations. They contain no option prices or transaction costs.

In [ ]:
decision_et = '2024-11-19 15:40' # @param {type:"string"}
display(charts.rich_pair(daily,bars,pairs,decision_et))

In [ ]:
# Build all six PNG archives. Each contains pairs.csv, a daily PNG per session and one intraday PNG per decision.
for year in range(2020,2026):
    saved = charts.year_images(out,year,daily,bars,pairs)
    print(saved.name,saved.stat().st_size,flush=True)
from google.colab import files
with zipfile.ZipFile(root/'SPY_paired_charts_2020_2025.zip','w',allowZip64=True) as package:
    for f in sorted(out.iterdir()):
        if f.is_file(): package.write(f,f.name)
files.download(str(root/'SPY_paired_charts_2020_2025.zip'))